# Microlensing follow-up prioritization

This notebook builds a seasonal target-selection table for El Leoncito. It combines every configured target source that is ready, MOP candidates visible in the requested season, local visibility, event parameters, historical survey coverage, and Rubin/DP2 coverage.

The final tables distinguish: source provenance, event parameters, visibility time, photometric points by microlensing stage, HSH image counts by stage, and DP2 visit/coadd coverage. Missing providers are reported explicitly and are not silently treated as observations.


## 1. Configure the observing season

Change only this cell for a new season or a different Rubin collection. HSH and JS source definitions are read from `configs/sources/`; JS may remain configured but disabled until its data file exists.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "observing_selection_summary.py").exists()), Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from target_selection import open_collection, report, target_io, visibility
from target_selection.prioritization import build_priority_table, empty_coverage
from target_selection.tasks.targets import collect_mop_targets
from observatory_observations import canonical_target_name, summarize_hsh_observations
from observing_selection_summary import build_observing_selection_summary

OBSERVATORY = "El Leoncito"
SEASON_START = "2026-09-10"
SEASON_END = "2026-09-12"
COLLECTION_NAME = "DP2"
EVENT_SOURCES = {"HSH": "hsh_events", "JS": "js_events"}
SOURCE_FILTERS = {"HSH": {}, "JS": {}}
MAGNITUDE_LIMITS = {"HSH": None, "JS": None, "MOP": None}
MIN_ALTITUDE_DEG = 30.0
MIN_VISIBLE_MINUTES = 30.0
OBSERVING_WINDOWS = ("01:30", "06:00")  # local time at El Leoncito
BANDS = tuple("ugrizy")
OUTPUT = REPO_ROOT / "notebooks" / "v2" / "outputs" / "microlensing_followup"
CACHE = OUTPUT / "cache"
OUTPUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
HSH_IMAGE_CATALOG = REPO_ROOT / "data" / "hsh_data" / "image_collection_astro.csv"
MOP_PHOTOMETRY_DIR = CACHE / "mop_photometry"
RUN_MOP_CANDIDATES = True
RUN_MOP_ENRICHMENT = True
RUN_RUBIN_MATCHING = True
RUBIN_WORKERS = 2


## 2. Check source availability

Configured CSV sources are checked for their definition, enabled flag and input file. MOP is checked as a provider: its seasonal candidate query is attempted in the next cell. A missing JS file therefore produces a status row and is skipped.


In [2]:
available_sources = set(target_io.list_sources())
source_rows = []
for survey, source in EVENT_SOURCES.items():
    definition = target_io.describe_source(source) if source in available_sources else {}
    raw_path = definition.get("path")
    path = Path(raw_path) if raw_path else None
    if path is not None and not path.is_absolute():
        path = Path(definition.get("base_dir", Path.cwd())) / path
    source_rows.append({"survey": survey, "source": source, "provider": "configured_catalog", "definition_exists": source in available_sources, "enabled": bool(definition.get("enabled", True)), "input_exists": bool(path and path.exists()), "input_path": str(path) if path else None})
source_rows.append({"survey": "MOP", "source": "MOP seasonal visibility", "provider": "mop_api", "definition_exists": True, "enabled": RUN_MOP_CANDIDATES, "input_exists": True, "input_path": "remote API"})
source_status = pd.DataFrame(source_rows)
source_status["ready"] = source_status["definition_exists"] & source_status["enabled"] & source_status["input_exists"]
source_status


,survey,source,provider,definition_exists,enabled,input_exists,input_path,ready
0,HSH,hsh_events,configured_catalog,True,True,True,/home/karennowo/target_selection/data/hsh_data...,True
1,JS,js_events,configured_catalog,True,False,False,/home/karennowo/target_selection/data/js_data/...,False
2,MOP,MOP seasonal visibility,mop_api,True,True,True,remote API,True


## 3. Load all available candidate events

The candidate population is the union of:

- configured HSH/JS target lists whose inputs are available;
- MOP targets returned as visible during the requested season.

Duplicates are merged by a canonical event name. MOP is preferred for coordinates and event parameters, while `input_sources` preserves all contributing sources.


In [3]:
event_frames = []
ready_sources = source_status.loc[source_status["provider"].eq("configured_catalog") & source_status["ready"]].set_index("survey")["source"].to_dict()
for survey, source in EVENT_SOURCES.items():
    if survey not in ready_sources:
        row = source_status.loc[source_status["survey"].eq(survey)].iloc[0]
        print(f"Skipping {survey}: configured={row['definition_exists']}, enabled={row['enabled']}, input_exists={row['input_exists']}.")
        continue
    source_filters = dict(SOURCE_FILTERS.get(survey, {}))
    if MAGNITUDE_LIMITS.get(survey) is not None:
        source_filters["magnitude"] = {"maximum": MAGNITUDE_LIMITS[survey]}
    frame = target_io.load(ready_sources[survey], filters=source_filters, verbose=True).copy()
    frame["survey"] = survey
    frame["input_sources"] = survey
    event_frames.append(frame)

if RUN_MOP_CANDIDATES:
    try:
        mop_legacy = collect_mop_targets(start_date=SEASON_START, end_date=SEASON_END, observatory=OBSERVATORY, include_event_data=True, cache_dir=CACHE, max_workers=4, verbose=True)
        if len(mop_legacy):
            mop_input = mop_legacy.rename(columns={"Target": "target_id", "RA_deg": "ra_deg", "Dec_deg": "dec_deg"})
            mop_filters = {}
            if MAGNITUDE_LIMITS.get("MOP") is not None:
                mop_filters["magnitude"] = {"column": "mag_now", "maximum": MAGNITUDE_LIMITS["MOP"]}
            mop_events = target_io.load(mop_input, source_name="MOP", filters=mop_filters, verbose=False)
            mop_events["survey"] = "MOP"
            mop_events["input_sources"] = "MOP"
            event_frames.append(mop_events)
        source_status.loc[source_status["survey"].eq("MOP"), "query_status"] = "loaded"
    except Exception as exc:
        print(f"MOP candidate query failed; continuing with configured catalogs: {exc}")
        source_status.loc[source_status["survey"].eq("MOP"), "query_status"] = "failed"
else:
    source_status.loc[source_status["survey"].eq("MOP"), "query_status"] = "disabled"

if event_frames:
    all_events = pd.concat(event_frames, ignore_index=True, sort=False)
    all_events["target_key"] = all_events["target_id"].map(canonical_target_name)
    all_events["_source_priority"] = all_events["survey"].map({"MOP": 0, "HSH": 1, "JS": 2}).fillna(9)
    provenance = all_events.groupby("target_key")["input_sources"].agg(lambda values: ",".join(sorted({item for value in values.fillna("").astype(str) for item in value.split(",") if item}))).rename("input_sources_all")
    events = all_events.sort_values(["target_key", "_source_priority"], kind="stable").drop_duplicates("target_key", keep="first").drop(columns=["_source_priority"]).merge(provenance, on="target_key", how="left")
    events["input_sources"] = events["input_sources_all"].fillna(events["input_sources"])
    events = events.drop(columns=["input_sources_all"]).reset_index(drop=True)
else:
    events = pd.DataFrame(columns=["target_id", "ra_deg", "dec_deg", "survey", "input_sources"])

source_status.to_csv(OUTPUT / "source_status.csv", index=False)
print(f"Unique candidate events: {len(events):,}")
source_status


Loaded 49 targets from hsh_events.
Skipping JS: configured=True, enabled=False, input_exists=False.
[targets] Querying MOP visibility for 2026-09-10 to 2026-09-12.
[targets] MOP targets with event data: 45
Unique candidate events: 92


,survey,source,provider,definition_exists,enabled,input_exists,input_path,ready,query_status
0,HSH,hsh_events,configured_catalog,True,True,True,/home/karennowo/target_selection/data/hsh_data...,True,NaN
1,JS,js_events,configured_catalog,True,False,False,/home/karennowo/target_selection/data/js_data/...,False,NaN
2,MOP,MOP seasonal visibility,mop_api,True,True,True,remote API,True,loaded


## 4. Add event parameters and cached photometry metadata

MOP enrichment adds canonical `magnitude`, `event_peak_hjd`, `event_duration_days` and `u0` columns. The MOP cache also stores per-target photometry, which is used later to count points in each event stage.


In [4]:
if RUN_MOP_ENRICHMENT and len(events):
    events = target_io.add_data(events, provider="MOP", fields=("magnitude", "event_duration_days", "event_peak_hjd", "u0"), provider_options={"cache_dir": CACHE / "mop_event_data", "photometry_dir": MOP_PHOTOMETRY_DIR, "max_workers": 4, "errors": "ignore"})
events["passes_magnitude"] = True
events.to_csv(OUTPUT / "all_candidate_events.csv", index=False)
events.head()


,target_id,ra_deg,dec_deg,ra_ecu,dec_ecu,source,survey,input_sources,RA,Dec,...,coordinate_offset_arcsec,coordinate_was_overridden,coordinate_source,coordinate_priority,target_key,magnitude,event_duration_days,event_peak_hjd,u0,passes_magnitude
0,Altair,297.695827,8.868321,19:50:46.99855,8:52:05.9563,hsh_events,HSH,HSH,NaN,NaN,...,NaN,NaN,NaN,NaN,ALTAIR,NaN,NaN,NaN,<NA>,True
1,ASASSN-24fs,296.495571,-7.924758,19:45:58.937,-07:55:29.13,hsh_events,HSH,HSH,NaN,NaN,...,NaN,NaN,NaN,NaN,ASASSN24FS,0.0,3.8176∓0.00623,2460558.25544∓0.00217,<NA>,True
2,Gaia22ahj,237.503070,-54.670480,NaN,NaN,MOP,MOP,MOP,15h50m00.7368s,-54d40m13.728s,...,NaN,False,MOP visibility table,90.0,GAIA22AHJ,NaN,NaN,NaN,<NA>,True
3,Gaia22dou,284.146820,-64.928720,NaN,NaN,MOP,MOP,MOP,18h56m35.2368s,-64d55m43.392s,...,NaN,False,MOP visibility table,90.0,GAIA22DOU,NaN,NaN,NaN,<NA>,True
4,Gaia22drk,282.611540,-46.716250,NaN,NaN,MOP,MOP,MOP,18h50m26.7696s,-46d42m58.5s,...,NaN,False,MOP visibility table,90.0,GAIA22DRK,NaN,NaN,NaN,<NA>,True


## 5. Evaluate El Leoncito visibility

Visibility is evaluated for every candidate event, not only for MOP candidates. The per-night table is retained, and the summary gives the number of selected nights, maximum observable minutes and the best local-time window.


In [5]:
if len(events):
    visibility_table = visibility.evaluate(events, start_date=SEASON_START, end_date=SEASON_END, observatory=OBSERVATORY, minimum_altitude_deg=MIN_ALTITUDE_DEG, minimum_observable_minutes=MIN_VISIBLE_MINUTES, observing_windows=OBSERVING_WINDOWS, output_path=OUTPUT / "visibility_by_night.csv", verbose=True)
    visibility_summary = visibility.summary(visibility_table)
    visibility_summary.to_csv(OUTPUT / "visibility_summary.csv", index=False)
    visibility.plots(events, start_date=SEASON_START, end_date=SEASON_END, output_dir=OUTPUT / "visibility_plots", observatory=OBSERVATORY, minimum_altitude_deg=MIN_ALTITUDE_DEG, minimum_observable_minutes=MIN_VISIBLE_MINUTES, observing_windows=OBSERVING_WINDOWS, selection=visibility_table, verbose=True)
else:
    visibility_table = pd.DataFrame()
    visibility_summary = pd.DataFrame()

visible_events = events.merge(visibility_summary[["target_id", "passes_visibility_filter", "visibility_nights_selected", "max_observable_minutes", "max_observable_night_percent", "best_observation_date", "best_observable_start_local", "best_observable_end_local"]], on="target_id", how="left") if len(events) else events.copy()
visible_events["passes_visibility_filter"] = visible_events["passes_visibility_filter"].astype("boolean").fillna(False).astype(bool)
visible_events["passes_magnitude"] = visible_events.get("passes_magnitude", True)
visible_events = visible_events.loc[visible_events["passes_visibility_filter"] & visible_events["passes_magnitude"]].copy()
visible_events.to_csv(OUTPUT / "visible_candidate_events.csv", index=False)
print(f"Visible candidates passing configured cuts: {len(visible_events):,}/{len(events):,}")


[visibility] Evaluating 92 targets from 2026-09-10 to 2026-09-12.
[visibility] Rows: 276; selected: 58.
      visibility: 1/3 (2026-09-10, 20/92 selected, 1 plot(s))
      visibility: 2/3 (2026-09-11, 19/92 selected, 1 plot(s))
      visibility: 3/3 (2026-09-12, 19/92 selected, 1 plot(s))
Visible candidates passing configured cuts: 20/92


### 6. Summarize event-stage coverage

The stage boundaries are expressed in Einstein times: rise, peak and fall are inside the event window, while pre/post-baseline are outside it. MOP photometry is counted as points; HSH is counted as usable astrometric images and also reported in exposure time. If JS data are not configured, its columns remain absent rather than being interpreted as zero observations.


In [6]:
legacy_visible = visible_events.rename(columns={"target_id": "Target", "ra_deg": "RA_deg", "dec_deg": "Dec_deg", "event_peak_hjd": "mop_t_0_hjd", "event_duration_days": "mop_t_e_days", "u0": "mop_u_0", "magnitude": "mag_now"}).copy()
legacy_visible = legacy_visible.loc[:, ~legacy_visible.columns.duplicated(keep="first")].copy()
for parameter_column in ("mop_t_0_hjd", "mop_t_e_days"):
    if parameter_column not in legacy_visible:
        legacy_visible[parameter_column] = np.nan
if len(legacy_visible):
    stage_summary = build_observing_selection_summary(legacy_visible, pd.DataFrame(), MOP_PHOTOMETRY_DIR, hsh_image_catalog=HSH_IMAGE_CATALOG if HSH_IMAGE_CATALOG.exists() else None, include_reference=False)
    hsh_summary = summarize_hsh_observations(legacy_visible, HSH_IMAGE_CATALOG) if HSH_IMAGE_CATALOG.exists() else pd.DataFrame(columns=["Target"])
    stage_summary = stage_summary.merge(hsh_summary, on="Target", how="outer", suffixes=("", "_hsh"))
    stage_summary = stage_summary.rename(columns={"Target": "target_id"})
    stage_summary = stage_summary.merge(visible_events[["target_id", "survey", "input_sources"]], on="target_id", how="left")
else:
    stage_summary = pd.DataFrame()
stage_summary.to_csv(OUTPUT / "survey_stage_summary.csv", index=False)
stage_summary.head()


,target_id,target_region,mag_now,visible_hours,best_observation_date,visible_from,visible_to,stage_classification_available,has_current_mop_data,t_E_days,...,hsh_post_baseline_n,hsh_unknown_n,hsh_n_I,hsh_exptime_I_s,hsh_n_V,hsh_exptime_V_s,hsh_n_N,hsh_exptime_N_s,survey,input_sources
0,ASASSN-24fs,UNK,NaN,0.533333,2026-09-10,01:30,02:01,False,False,NaN,...,0,519,278,19980.0,222,16110.0,19,2820.0,HSH,HSH
1,Gaia22dou,UNK,NaN,1.516667,2026-09-10,01:30,03:00,False,False,NaN,...,0,0,0,NaN,0,NaN,0,NaN,MOP,MOP
2,Gaia22drk,UNK,NaN,0.883333,2026-09-10,01:30,02:22,False,False,NaN,...,0,0,0,NaN,0,NaN,0,NaN,MOP,MOP
3,Gaia22eiw,UNK,NaN,2.883333,2026-09-12,03:08,06:00,False,False,NaN,...,0,0,0,NaN,0,NaN,0,NaN,MOP,MOP
4,Gaia23cuh,UNK,NaN,3.333333,2026-09-10,01:30,04:49,False,False,NaN,...,0,0,0,NaN,0,NaN,0,NaN,MOP,MOP


## 7. Query Rubin/DP2 coverage

Only candidates that pass the current visibility and magnitude criteria are sent to Rubin matching. Visit coverage is obtained through the collection's spatially tiled TAP query; coadd coverage is then checked with Butler, and property-map values are sampled at each target position.


In [ ]:
if RUN_RUBIN_MATCHING and len(visible_events):
    collection = open_collection(COLLECTION_NAME)
    coverage = collection.get_visit_coverage(visible_events, bands=BANDS, cache_dir=CACHE / "visits", max_workers=RUBIN_WORKERS)
    coverage = collection.get_coadd_coverage(coverage, bands=BANDS, cache_path=CACHE / "coadds.csv")
    positions = coverage[["target_id", "ra_deg", "dec_deg"]].drop_duplicates("target_id")
    properties = collection.get_property_map_values(positions, properties=("maglim", "psf_size", "exposure_time", "sky_noise"), bands=BANDS, cache_path=CACHE / "properties.csv")
    coverage = coverage.merge(properties, on="target_id", how="left", suffixes=("", "_map"))
else:
    coverage = empty_coverage(visible_events)
coverage.to_csv(OUTPUT / "rubin_dp2_coverage.csv", index=False)
coverage.head()


Visit tiles: 100%|██████████| 19/19 [00:00<00:00, 172.74tile/s]

Visit coverage: 19 spatial tiles, 20 targets, 41 target-band rows


Coadd map: maglim u (1/20 positions)
Coadd map: maglim g (2/20 positions)
Coadd map: maglim r (2/20 positions)
Coadd map: maglim i (2/20 positions)
Coadd map: maglim z (1/20 positions)


/home/karennowo/target_selection/target_selection/rsp_access.py:80: RuntimeWarning: Partial map read failed for deepCoadd_psf_maglim_consolidated_map_weighted_mean y; falling back to Butler.get: None of the specified pixels are in the coverage map.
  result = sample_coadd_properties(


## 8. Build the review table and priority score

This table is the main decision product. It preserves the event parameters, source provenance, visibility duration, stage-by-stage history and DP2 coverage. The score is only an auditable starting point; the component columns are retained for scientific review.


In [ ]:
mop_point_columns = [column for column in stage_summary.columns if column.startswith("mop_") and column.endswith("_points")] if len(stage_summary) else []
if len(stage_summary) and mop_point_columns:
    stage_summary["mop_photometry_points_total"] = stage_summary[mop_point_columns].sum(axis=1)
    mop_photometry_ids = set(stage_summary.loc[stage_summary["mop_photometry_points_total"].gt(0), "target_id"].astype(str))
else:
    if len(stage_summary):
        stage_summary["mop_photometry_points_total"] = 0
    mop_photometry_ids = set()

photometry_summary = stage_summary[["target_id"]].copy() if len(stage_summary) else pd.DataFrame(columns=["target_id"])
if len(photometry_summary):
    photometry_summary["n_photometry"] = stage_summary["mop_photometry_points_total"]
    pre_points = pd.to_numeric(stage_summary.get("mop_pre_baseline_points", pd.Series(0, index=stage_summary.index)), errors="coerce").fillna(0)
    post_points = pd.to_numeric(stage_summary.get("mop_post_baseline_points", pd.Series(0, index=stage_summary.index)), errors="coerce").fillna(0)
    rise_points = pd.to_numeric(stage_summary.get("mop_rise_points", pd.Series(0, index=stage_summary.index)), errors="coerce").fillna(0)
    peak_points = pd.to_numeric(stage_summary.get("mop_peak_points", pd.Series(0, index=stage_summary.index)), errors="coerce").fillna(0)
    fall_points = pd.to_numeric(stage_summary.get("mop_fall_points", pd.Series(0, index=stage_summary.index)), errors="coerce").fillna(0)
    photometry_summary["has_baseline"] = pre_points.gt(0) | post_points.gt(0)
    photometry_summary["has_rise"] = rise_points.gt(0)
    photometry_summary["has_peak"] = peak_points.gt(0)
    photometry_summary["has_decline"] = fall_points.gt(0)
    photometry_summary["event_state"] = np.where(photometry_summary["has_decline"], "decline_observed", np.where(photometry_summary["has_peak"], "peak_observed", "incomplete"))

priority_events = visible_events.loc[visible_events["target_id"].astype(str).isin(mop_photometry_ids)].copy()
priority = build_priority_table(priority_events, visibility_table, photometry_summary, coverage) if len(priority_events) else pd.DataFrame()
if len(priority):
    priority = priority.merge(visibility_summary, on="target_id", how="left", suffixes=("", "_visibility"))
    priority = priority.merge(stage_summary, on="target_id", how="left", suffixes=("", "_stages"))
    priority["mop_photometry_available"] = True
    preferred_columns = [
        "target_id", "survey", "input_sources", "magnitude", "event_peak_hjd", "event_duration_days", "u0",
        "visibility_nights_selected", "max_observable_minutes", "max_observable_night_percent",
        "best_observation_date", "best_observable_start_local", "best_observable_end_local",
        "mop_photometry_points_total", *mop_point_columns,
        "hsh_n_usable", "hsh_pre_baseline_n", "hsh_rise_n", "hsh_peak_n", "hsh_fall_n", "hsh_post_baseline_n",
        "has_visit_coverage", "has_coadd", "priority_score", "priority_reason",
    ]
    priority = priority[[column for column in preferred_columns if column in priority] + [column for column in priority.columns if column not in preferred_columns]]
priority.to_csv(OUTPUT / "priority_table.csv", index=False)
priority.head(20)


## 9. Diagnostic plots

These plots separate the decision inputs: source membership, visibility time, historical phase coverage and DP2 availability.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
if len(priority):
    priority["priority_score"].plot.hist(ax=axes[0, 0], bins=10, color="tab:blue")
    axes[0, 0].set(title="Priority score", xlabel="Score", ylabel="Targets")
    priority["max_observable_minutes"].plot.hist(ax=axes[0, 1], bins=10, color="tab:green")
    axes[0, 1].set(title="Maximum observable time", xlabel="Minutes", ylabel="Targets")
    stage_columns = [c for c in priority.columns if c.startswith("mop_") and c.endswith("_points")]
    if stage_columns:
        priority[stage_columns].sum().plot.bar(ax=axes[1, 0], color="tab:purple")
        axes[1, 0].set(title="MOP points by event stage", ylabel="Points")
    dp2_counts = pd.Series({"Visits": int(priority.get("has_visit_coverage", pd.Series(False, index=priority.index)).astype(bool).sum()), "Coadds": int(priority.get("has_coadd", pd.Series(False, index=priority.index)).astype(bool).sum())})
    dp2_counts.plot.bar(ax=axes[1, 1], color="tab:orange")
    axes[1, 1].set(title="DP2 coverage", ylabel="Targets")
else:
    for axis in axes.flat: axis.text(.5, .5, "No visible candidates", ha="center", va="center")
fig.tight_layout()
fig.savefig(OUTPUT / "priority_diagnostics.png", dpi=180)
plt.show()


## 10. Optional Rubin products

Run this cell only after the tables are correct. It creates the standard matching plots and event cutouts for the visible candidate set.


In [ ]:
if RUN_RUBIN_MATCHING and len(coverage):
    report.match_summary(coverage, OUTPUT, bands=BANDS, category_column="survey", category_label="Survey", name="microlensing_followup")
    report.property_distributions(coverage, OUTPUT, bands=BANDS, category_column="survey", category_label="Survey", name="microlensing_followup")
    report.sky_maps(coverage, OUTPUT, bands=BANDS, name="microlensing_followup")
    collection.get_image(coverage, image_type="coadd", output_dir=OUTPUT / "cutouts" / "pngs", fits_dir=OUTPUT / "cutouts" / "fits", bands=BANDS, category_column="survey", add_rgb=False)


## Interpretation and provider status

`source_status.csv`, `visible_candidate_events.csv`, `visibility_summary.csv`, `survey_stage_summary.csv`, `rubin_dp2_coverage.csv` and `priority_table.csv` are the auditable outputs. HSH stage counts are based on the local astrometric image inventory; MOP stage counts use cached MOP photometry; JS/OGLE/OMP are included only when their data and adapters are configured.
